In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from timm import create_model
from PIL import Image
import pandas as pd
import os
from sklearn.model_selection import train_test_split

df = pd.read_csv("train.csv")

c:\Akshat\ML-Projects\Retinopathy\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
label_map = {
    0: "No_DR",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferate_DR"
}

In [5]:
class RetinopathyDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root = root_dir
        self.transform = transform

        self.num_to_str = {
            0: "No_DR",
            1: "Mild",
            2: "Moderate",
            3: "Severe",
            4: "Proliferate_DR"
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_name = row["id_code"] + ".png"
        label_num = int(row["diagnosis"])    

        folder_name = self.num_to_str[label_num] 

        img_path = os.path.join(self.root, folder_name, img_name)

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label_num, dtype=torch.long)


In [6]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["diagnosis"], random_state=42)

In [7]:
from torchvision import transforms
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [8]:
from torch.utils.data import DataLoader
train_ds = RetinopathyDataset(train_df, root_dir="colored_images", transform=train_transforms)
val_ds   = RetinopathyDataset(val_df, root_dir="colored_images", transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = create_model("efficientnet_b0", pretrained=True)
model.classifier = nn.Linear(model.classifier.in_features, 5)
model = model.to(device)

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [13]:
best_val_acc = 0.0
best_model_path = "backend/models/best_efficientnet.pth"

In [15]:
from tqdm import tqdm

def train_model(model, train_loader, val_loader, criterion, optimizer, device, best_val_acc, best_model_path, epochs=10):

    for epoch in range(epochs):
        print(f"\n========== Epoch {epoch+1}/{epochs} ==========")

        model.train()
        running_loss = 0
        correct = 0
        total = 0

        for imgs, labels in tqdm(train_loader, desc="Training"):
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        print(f"Train Loss: {running_loss/len(train_loader):.4f}, "
              f"Train Acc: {correct/total:.4f}")


        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for imgs, labels in tqdm(val_loader, desc="Validating"):
                imgs, labels = imgs.to(device), labels.to(device)

                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        

        print(f"Val Loss: {val_loss/len(val_loader):.4f}, "
              f"Val Acc: {val_correct/val_total:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
        
    return model


In [16]:
trained_model = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    best_val_acc=best_val_acc,
    best_model_path=best_model_path,
    epochs=10
)



========== Epoch 1/10 ==========


Training: 100%|██████████| 92/92 [00:58<00:00,  1.57it/s]


Train Loss: 0.8419, Train Acc: 0.7067


Validating: 100%|██████████| 23/23 [00:08<00:00,  2.75it/s]


Val Loss: 0.6210, Val Acc: 0.7653

========== Epoch 2/10 ==========


...

Val Loss: 0.6566, Val Acc: 0.8022

========== Epoch 10/10 ==========


Training: 100%|██████████| 92/92 [01:35<00:00,  1.04s/it]


Train Loss: 0.1035, Train Acc: 0.9679


Validating: 100%|██████████| 23/23 [00:09<00:00,  2.50it/s]

Val Loss: 0.7884, Val Acc: 0.8117


In [4]:
from pinecone import Pinecone as PineconeClient, ServerlessSpec
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

In [2]:
from dotenv import load_dotenv
load_dotenv()


True

In [6]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

index_name = "retinopathy"
vector_store = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [7]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [8]:
docs = retriever.invoke("What is diabetic retinopathy?")
print(docs)

[Document(id='9141d65d-e1db-4ebd-ad80-481d96e158b7', metadata={'creationdate': '2016-08-04T12:03:10-05:00', 'creator': 'Adobe InDesign CC 2014 (Macintosh)', 'moddate': '2016-08-04T12:03:12-05:00', 'page': 2.0, 'page_label': '3', 'producer': 'Adobe PDF Library 11.0', 'source': '../rag/fact_sheet_22_diabetic_retinopathy_new.pdf', 'total_pages': 4.0, 'trapped': '/False'}, page_content='retinopathy or diabetes and have vision loss that cannot be reversed,  \na retina specialist can help you find access to rehabilitation with a  \nvariety of tools to make everyday living with this disease a little bit  \neasier. A retina specialist can also help connect you with others who  \nhave similar limitations.\nDiabetic Retinopathy  \ncontinued from previous page \nCopyright 2016 The Foundation of the American Society of Retina Specialists. All rights reserved.savingvision.org  I  20 North Wacker Drive, Suite 2030, Chicago, IL 60606  |  (312) 578-8760\nFigure 3\nFA of a patient with proliferative di

In [14]:
docs[2].page_content

'Diabetic Retinopathy: Diabetic retinopathy \n(pronounced ret in OP uh thee) is a complication of diabetes \nthat causes damage to the blood vessels of the retina— \nthe light-sensitive tissue that lines the back part of the eye, \nallowing you to see fine detail.  \nAmerican Society of Retina Specialists\nThe FoundationRETINA HEALTH SERIES  |  Facts from the ASRS\nCommitted to improving  \nthe quality of life of all people  \nwith retinal disease. \nCopyright 2016 The Foundation of the American Society of Retina Specialists. All rights reserved.savingvision.org  I  20 North Wacker Drive, Suite 2030, Chicago, IL 60606  |  (312) 578-8760\nDiabetic retinopathy is the most common cause of irreversible blindness in \nworking-age Americans. As many people with type 1 diabetes suffer blindness \nas those with the more common type 2 disease. Diabetic retinopathy occurs \nin more than half of the people who develop diabetes. \nCauses: The primary cause of diabetic retinopathy is diabetes—a con